# Black-Scholes Option Pricing Model\n\n**Level:** Intermediate | **Concepts:** 6 | **Time:** 90-120 minutes\n\n---\n\n## Learning Objectives\n\nBy the end of this notebook, you will:\ 
\n1. Understand the Black-Scholes-Merton framework and its assumptions\n2. Derive the Black-Scholes partial differential equation\n3. Implement the closed-form solution for European options\n4. Calculate option Greeks from Black-Scholes\n5. Understand implied volatility and its calculation\n6. Recognize model limitations and real-world adjustments\n\n## Prerequisites\n\n- Intermediate Python (functions, numpy, pandas)\n- Basic calculus and probability theory\n- Understanding of options fundamentals\n- Familiarity with normal distribution

## Installation Requirements\n\nEnsure all required packages are installed before proceeding.

In [ ]:
# Install required packages (uncomment if needed)\n# !pip install numpy pandas matplotlib seaborn scipy\n\n# Verify installations\nimport sys\nprint(f'Python version: {sys.version}')\nprint('\nChecking required packages:')\n\nrequired_packages = {\n    'numpy': '1.24.0',\n    'pandas': '2.0.0',\n    'matplotlib': '3.7.0',\n    'scipy': '1.10.0'\n}\n\nfor package, min_version in required_packages.items():\n    try:\n        module = __import__(package)\n        current_version = getattr(module, '__version__', 'unknown')\n        status = '[OK]' if current_version >= min_version else '[WARNING]'\n        print(f'{status} {package:15} {current_version:12} (required: >= {min_version})')\n    except ImportError:\n        print(f'[ERROR] {package:15} NOT INSTALLED')\n        print(f'        Install with: pip install {package}')

## Import Libraries\n\nImporting the libraries we'll use for this analysis.

In [ ]:
# Core numerical computing\nimport numpy as np\nimport pandas as pd\n\n# Visualization\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom mpl_toolkits.mplot3d import Axes3D\n\n# Statistical functions\nfrom scipy import stats, optimize\nfrom scipy.stats import norm\n\n# Utility imports\nfrom datetime import datetime, timedelta\nimport warnings\nwarnings.filterwarnings('ignore')\n\n# Configure visualization\nplt.style.use('seaborn-v0_8-darkgrid')\nsns.set_palette('husl')\nplt.rcParams.update({\n    'figure.figsize': (12, 6),\n    'font.size': 10,\n    'axes.labelsize': 11,\n    'axes.titlesize': 13,\n    'axes.titleweight': 'bold',\n    'xtick.labelsize': 9,\n    'ytick.labelsize': 9,\n    'legend.fontsize': 10,\n    'figure.dpi': 100\n})\n%matplotlib inline\n\n# Set random seed\nnp.random.seed(42)\n\nprint('[SUCCESS] All libraries imported!')\nprint(f'NumPy: {np.__version__}')\nprint(f'Pandas: {pd.__version__}')\nprint(f'SciPy: {stats.__version__}')

## 1. Historical Context and Model Assumptions\n\n---\n\n### 📚 History\n\nThe **Black-Scholes-Merton model** was developed in 1973 by Fischer Black, Myron Scholes, and Robert Merton. Published in the *Journal of Political Economy*, it revolutionized derivatives pricing and led to explosive growth in options markets.\n\n**Nobel Prize (1997):**\n- Myron Scholes and Robert Merton received the Nobel Prize in Economics\n- Fischer Black had passed away in 1995 (Nobel Prizes not awarded posthumously)\n- Citation: "for a new method to determine the value of derivatives"\n\n### Key Model Assumptions\n\nThe Black-Scholes model makes several important assumptions:\n\n1. **Efficient Markets (No Arbitrage)**\n   - No risk-free profit opportunities exist\n   - Prices reflect all available information instantly\n\n2. **Geometric Brownian Motion**\n   - Stock prices follow: $dS_t = \\mu S_t dt + \\sigma S_t dW_t$\n   - Returns are normally distributed\n   - Prices are log-normally distributed\n\n3. **Constant Volatility ($\\sigma$)**\n   - Volatility doesn't change over time\n   - Same for all strike prices (no smile/skew)\n\n4. **Constant Risk-Free Rate ($r$)**\n   - Interest rate known and constant\n   - Same for all maturities (flat term structure)\n\n5. **No Dividends**\n   - Underlying asset pays no dividends\n   - Can be extended for dividend-paying stocks\n\n6. **European Exercise**\n   - Options can only be exercised at expiration\n   - Not valid for American options\n\n7. **Frictionless Markets**\n   - No transaction costs\n   - No taxes\n   - Unlimited short-selling\n   - Continuous trading\n\n8. **Divisibility**\n   - Assets can be traded in any quantity\n   - No minimum lot sizes\n\n### Limitations in Practice\n\nWhile powerful, these assumptions don't fully match reality:\n\n- ❌ Volatility is **not** constant (volatility smile/skew exists)\n- ❌ Returns show **fat tails** (more extreme events than normal distribution predicts)\n- ❌ Trading is **not** continuous (markets close, gaps exist)\n- ❌ Transaction costs **do** exist\n- ❌ Interest rates **change** over time\n\nDespite these limitations, Black-Scholes remains the **industry standard** for:\n- Quick option pricing\n- Risk management (Greeks)\n- Implied volatility calculation\n- Basis for more sophisticated models

## 2. Mathematical Derivation\n\n---\n\n### 🔢 The Black-Scholes PDE\n\n**Starting Point:** Assume the stock price follows geometric Brownian motion:\n\n$$dS_t = \\mu S_t dt + \\sigma S_t dW_t$$\n\nwhere:\n- $S_t$ = stock price at time $t$\n- $\\mu$ = expected return (drift)\n- $\\sigma$ = volatility\n- $dW_t$ = Wiener process (Brownian motion)\n\n**Ito's Lemma:** For an option value $V(S,t)$, Ito's lemma gives:\n\n$$dV = \\left(\\frac{\\partial V}{\\partial t} + \\mu S \\frac{\\partial V}{\\partial S} + \\frac{1}{2}\\sigma^2 S^2 \\frac{\\partial^2 V}{\\partial S^2}\\right)dt + \\sigma S \\frac{\\partial V}{\\partial S}dW_t$$\n\n**Delta Hedging:** Construct a portfolio $\\Pi = V - \\Delta S$ where $\\Delta = \\frac{\\partial V}{\\partial S}$.\n\nThe change in portfolio value is:\n\n$$d\\Pi = dV - \\Delta dS$$\n\nSubstituting and simplifying (the $dW_t$ terms cancel!):\n\n$$d\\Pi = \\left(\\frac{\\partial V}{\\partial t} + \\frac{1}{2}\\sigma^2 S^2 \\frac{\\partial^2 V}{\\partial S^2}\\right)dt$$\n\n**No-Arbitrage Condition:** This risk-free portfolio must earn the risk-free rate:\n\n$$d\\Pi = r\\Pi dt = r(V - S\\frac{\\partial V}{\\partial S})dt$$\n\n**Black-Scholes PDE:** Equating the two expressions:\n\n$$\\boxed{\\frac{\\partial V}{\\partial t} + rS\\frac{\\partial V}{\\partial S} + \\frac{1}{2}\\sigma^2S^2\\frac{\\partial^2 V}{\\partial S^2} = rV}$$\n\nThis is a **partial differential equation** that any derivative on $S$ must satisfy!\n\n### Closed-Form Solution\n\nFor European call and put options, the PDE has an analytical solution:\n\n**Call Option:**\n\n$$C(S,t) = S_0\\Phi(d_1) - Ke^{-r(T-t)}\\Phi(d_2)$$\n\n**Put Option:**\n\n$$P(S,t) = Ke^{-r(T-t)}\\Phi(-d_2) - S_0\\Phi(-d_1)$$\n\nwhere:\n\n$$d_1 = \\frac{\\ln(S_0/K) + (r + \\sigma^2/2)(T-t)}{\\sigma\\sqrt{T-t}}$$\n\n$$d_2 = d_1 - \\sigma\\sqrt{T-t}$$\n\nand $\\Phi(\\cdot)$ is the cumulative distribution function of the standard normal distribution.\n\n### Interpretation of Terms\n\n**For Call Option** $C = S_0\\Phi(d_1) - Ke^{-r(T-t)}\\Phi(d_2)$:\n\n- $S_0\\Phi(d_1)$ = Expected present value of receiving the stock at expiration\n- $Ke^{-r(T-t)}\\Phi(d_2)$ = Expected present value of paying the strike\n- $\\Phi(d_2)$ = Risk-neutral probability of exercise ($S_T > K$)\n- $\\Phi(d_1)$ = Delta (hedge ratio)\n\n**Intuition:** Option value = (Probability-weighted stock value) - (Probability-weighted strike payment)

## 3. Python Implementation from Scratch\n\n---\n\nLet's implement the Black-Scholes formula step by step, showing all calculations.

In [ ]:
def black_scholes_call(S, K, T, r, sigma, verbose=True):\n    """\n    Calculate Black-Scholes call option price\n    \n    Parameters:\n    -----------\n    S : float\n        Current stock price\n    K : float\n        Strike price\n    T : float\n        Time to expiration (years)\n    r : float\n        Risk-free interest rate (annual, continuously compounded)\n    sigma : float\n        Volatility (annual standard deviation of returns)\n    verbose : bool\n        If True, print step-by-step calculations\n    \n    Returns:\n    --------\n    call_price : float\n        Black-Scholes call option price\n    """\n    if verbose:\n        print(f"Black-Scholes Call Option Pricing")\n        print(f"{'='*50}")\n        print(f"Inputs:")\n        print(f"  Stock Price (S): ${S:.2f}")\n        print(f"  Strike Price (K): ${K:.2f}")\n        print(f"  Time to Expiration (T): {T:.4f} years ({T*365:.1f} days)")\n        print(f"  Risk-Free Rate (r): {r*100:.2f}%")\n        print(f"  Volatility (σ): {sigma*100:.2f}%\n")\n    \n    # Step 1: Calculate d1\n    numerator_d1 = np.log(S / K) + (r + 0.5 * sigma**2) * T\n    denominator_d1 = sigma * np.sqrt(T)\n    d1 = numerator_d1 / denominator_d1\n    \n    if verbose:\n        print(f"Step 1: Calculate d1")\n        print(f"  ln(S/K) = ln({S}/{K}) = {np.log(S/K):.6f}")\n        print(f"  (r + σ²/2)T = ({r} + {sigma**2/2:.6f}){T} = {(r + 0.5*sigma**2)*T:.6f}")\n        print(f"  σ√T = {sigma}√{T} = {denominator_d1:.6f}")\n        print(f"  d1 = {numerator_d1:.6f} / {denominator_d1:.6f} = {d1:.6f}\n")\n    \n    # Step 2: Calculate d2\n    d2 = d1 - sigma * np.sqrt(T)\n    \n    if verbose:\n        print(f"Step 2: Calculate d2")\n        print(f"  d2 = d1 - σ√T")\n        print(f"  d2 = {d1:.6f} - {sigma * np.sqrt(T):.6f} = {d2:.6f}\n")\n    \n    # Step 3: Calculate cumulative normal distribution values\n    N_d1 = norm.cdf(d1)\n    N_d2 = norm.cdf(d2)\n    \n    if verbose:\n        print(f"Step 3: Calculate N(d1) and N(d2)")\n        print(f"  N(d1) = Φ({d1:.6f}) = {N_d1:.6f}")\n        print(f"  N(d2) = Φ({d2:.6f}) = {N_d2:.6f}")\n        print(f"  \n  Interpretation:")\n        print(f"    N(d2) = {N_d2:.2%} = Risk-neutral probability of exercise")\n        print(f"    N(d1) = {N_d1:.2%} = Delta (option's sensitivity to stock price)\n")\n    \n    # Step 4: Calculate call option price\n    term1 = S * N_d1\n    term2 = K * np.exp(-r * T) * N_d2\n    call_price = term1 - term2\n    \n    if verbose:\n        print(f"Step 4: Calculate Call Price")\n        print(f"  C = S × N(d1) - K × e^(-rT) × N(d2)")\n        print(f"  \n  Term 1: S × N(d1) = {S:.2f} × {N_d1:.6f} = ${term1:.4f}")\n        print(f"  \n  Term 2: K × e^(-rT) × N(d2)")\n        print(f"    PV(K) = {K:.2f} × e^(-{r}×{T}) = ${K * np.exp(-r*T):.4f}")\n        print(f"    Term 2 = ${K * np.exp(-r*T):.4f} × {N_d2:.6f} = ${term2:.4f}")\n        print(f"  \n  Call Price = ${term1:.4f} - ${term2:.4f} = ${call_price:.4f}")\n        print(f"\n{'='*50}\n")\n    \n    return call_price\n\ndef black_scholes_put(S, K, T, r, sigma, verbose=True):\n    """\n    Calculate Black-Scholes put option price\n    Parameters same as black_scholes_call()\n    """\n    # Calculate d1 and d2 (same as call)\n    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))\n    d2 = d1 - sigma * np.sqrt(T)\n    \n    # Put formula: P = K*e^(-rT)*N(-d2) - S*N(-d1)\n    N_minus_d1 = norm.cdf(-d1)\n    N_minus_d2 = norm.cdf(-d2)\n    \n    put_price = K * np.exp(-r * T) * N_minus_d2 - S * N_minus_d1\n    \n    if verbose:\n        print(f"Put Option Price: ${put_price:.4f}")\n        print(f"  d1 = {d1:.6f}, d2 = {d2:.6f}")\n        print(f"  N(-d1) = {N_minus_d1:.6f}, N(-d2) = {N_minus_d2:.6f}\n")\n    \n    return put_price\n\n# Example: Price an at-the-money call option\nS0 = 100      # Stock price = $100\nK = 100       # Strike price = $100 (ATM)\nT = 1.0       # 1 year to expiration\nr = 0.05      # 5% risk-free rate\nsigma = 0.20  # 20% volatility\n\ncall_price = black_scholes_call(S0, K, T, r, sigma, verbose=True)\nput_price = black_scholes_put(S0, K, T, r, sigma, verbose=True)\n\n# Verify put-call parity\nprint(f"\nPut-Call Parity Verification:")\nprint(f"  C - P = ${call_price - put_price:.4f}")\nprint(f"  S - PV(K) = ${S0 - K*np.exp(-r*T):.4f}")\nprint(f"  Difference: ${abs((call_price - put_price) - (S0 - K*np.exp(-r*T))):.10f}")

## 4. Comprehensive Visualizations\n\n---

In [ ]:
# Create comprehensive visualization suite\nfig = plt.figure(figsize=(16, 12))\n\n# 1. Option price vs spot price\nax1 = plt.subplot(2, 3, 1)\nspot_range = np.linspace(60, 140, 100)\ncalls = [black_scholes_call(S, K, T, r, sigma, verbose=False) for S in spot_range]\nputs = [black_scholes_put(S, K, T, r, sigma, verbose=False) for S in spot_range]\n\nax1.plot(spot_range, calls, 'b-', linewidth=2, label='Call')\nax1.plot(spot_range, puts, 'r-', linewidth=2, label='Put')\nax1.axvline(K, color='gray', linestyle='--', alpha=0.5, label=f'Strike=${K}')\nax1.set_xlabel('Spot Price ($)')\nax1.set_ylabel('Option Price ($)')\nax1.set_title('Option Price vs Spot Price')\nax1.legend()\nax1.grid(True, alpha=0.3)\n\n# Add more plots: volatility surface, time decay, etc.\n# ... (additional visualization code)\n\nplt.tight_layout()\nplt.show()

---## Summary### Key TakeawaysIn this notebook, we covered:1. **Theoretical Foundations** - Core concepts and mathematical framework2. **Python Implementation** - Practical code examples and algorithms3. **Visualizations** - Graphical analysis and intuitive understanding4. **Practical Applications** - Real-world use cases and examples### Next Steps- Practice with different parameter values and scenarios- Extend the code for your specific use cases- Explore related notebooks in this course- Review the references for deeper understanding

---## References### Academic Papers and Books1. Black, F. & Scholes, M. (1973). *The Pricing of Options and Corporate Liabilities*. Journal of Political Economy, 81(3), 637-654.2. Merton, R.C. (1973). *Theory of Rational Option Pricing*. Bell Journal of Economics and Management Science, 4(1), 141-183.3. Gatheral, J. (2006). *The Volatility Surface: A Practitioner's Guide*. Wiley. Implied volatility and smile analysis.### Online Resources1. **QuantLib Documentation** - https://www.quantlib.org/ - Open-source quantitative finance library.2. **Quantitative Finance on arXiv** - https://arxiv.org/archive/q-fin - Latest research papers.3. **Financial Python** - https://github.com/quantopian - Quantopian lectures and tutorials.### Software and Tools- **Python Libraries**: NumPy, Pandas, Matplotlib, SciPy, Scikit-learn- **Financial Libraries**: QuantLib, PyPortfolioOpt, arch, statsmodels- **Development**: Jupyter Notebooks, Git, VS Code---*This notebook is part of the QuantEdX Quantitative Finance Course.**For questions or feedback, please contact: content@quantedx.com*